## 1. Bibliotecas

In [2]:
import os
import re
import pandas as pd

## 2. Caminhos dos arquivos

Caminhos para os arquivos de dados brutos do Fake.Br e do Fake.Br-LLM.

In [3]:
FAKEBR_PATH = 'data/raw/Fake_Br/full_texts/fake'

In [4]:
TRAIN_FAKEBR_LLM_PATH = 'data/raw/Fake.Br-LLM/train-fake-LLM'
TEST_FAKEBR_LLM_PATH = 'data/raw/Fake.Br-LLM/test-fake-LLM'

## 3. Funções de carregamento

Funções para ler e extrair o conteúdo dos arquivos de cada fonte de dados.

In [5]:
def read_news_fakebr(file_path):
  """
  Lê um arquivo de notícia falsa do Fake.Br Corpus.

  Parâmetros:
    file_path (str): caminho completo do arquivo .txt

  Retorna:
    dict com 'id' (extraído do nome do arquivo) e 'text' (conteúdo do arquivo)
  """

  with open(file_path, 'r', encoding='utf-8') as file:
    text = file.read()

  file_name = file_path.split('/')[-1]
  id_news = file_name.replace('.txt', '')

  return {'id': id_news, 'text': text.strip()}

In [6]:
def read_news_fakebr_llm(file_path):
  """
  Lê um arquivo de notícia do Fake.Br-LLM Corpus.

  Parâmetros:
    file_path (str): caminho completo do arquivo .txt

  Retorna:
    dict com 'id' (extraído do nome do arquivo), 'original_news' (texto verdadeiro original),
    'synthetic_news' (texto falso gerado por LLM) e 'llm_changes' (explicação das mudanças
    feitas pelo LLM, usada apenas para inspeção)
  """

  with open(file_path, 'r', encoding='utf-8') as file:
    text = file.read()

  file_name = file_path.split('/')[-1]
  id_news = file_name.replace('.txt', '')

  match_original = re.search('<originalText>(.*?)</originalText>', text, re.DOTALL)
  match_synthetic = re.search('<syntheticText>(.*?)</syntheticText>', text, re.DOTALL)
  match_changes = re.search('<changes>(.*?)</changes>', text, re.DOTALL)

  if match_original is None:
    original_news = ''
  else:
    original_news = match_original.group(1)

  if match_synthetic is None:
    synthetic_news = ''
  else:
    synthetic_news = match_synthetic.group(1)

  if match_changes is None:
    llm_changes = ''
  else:
    llm_changes = match_changes.group(1)

  return {'id': id_news, 'original_news': original_news.strip(), 'synthetic_news': synthetic_news.strip(), 'llm_changes': llm_changes.strip()}


## 4. Carregamento dos dados

Aplicação das funções de leitura sobre todos os arquivos de cada pasta, gerando as listas de notícias.

In [10]:
human_fakebr_news_raw = []

for file_name in os.listdir(FAKEBR_PATH):
  full_path = os.path.join(FAKEBR_PATH, file_name)
  news = read_news_fakebr(full_path)
  human_fakebr_news_raw.append(news)

print(f'Total de notícias carregadas: {len(human_fakebr_news_raw)}')

Total de notícias carregadas: 3600


In [11]:
train_fakebr_llm_raw = []

for file_name in os.listdir(TRAIN_FAKEBR_LLM_PATH):
  full_path = os.path.join(TRAIN_FAKEBR_LLM_PATH, file_name)
  synthetic_news = read_news_fakebr_llm(full_path)
  train_fakebr_llm_raw.append(synthetic_news)

print(f'Total de notícias para treinamento carregadas: {len(train_fakebr_llm_raw)}')

Total de notícias para treinamento carregadas: 2880


In [12]:
test_fakebr_llm_raw = []

for file_name in os.listdir(TEST_FAKEBR_LLM_PATH):
  full_path = os.path.join(TEST_FAKEBR_LLM_PATH, file_name)
  synthetic_news = read_news_fakebr_llm(full_path)
  test_fakebr_llm_raw.append(synthetic_news)

print(f'Total de notícias para teste carregadas: {len(test_fakebr_llm_raw)}')

Total de notícias para teste carregadas: 720


## 5. Consolidação

União das notícias humanas e sintéticas em um único DataFrame, seguindo a estrutura de colunas definida (id, text, label, origin, source, split, llm_explanation).

In [13]:
train_ids = set()
test_ids = set()

for item in train_fakebr_llm_raw:
  train_ids.add(item['id'])

for item in test_fakebr_llm_raw:
  test_ids.add(item['id'])

In [14]:
rows_fakebr = []

for news in human_fakebr_news_raw:
  if news['id'] in train_ids:
    split = 'train'
  elif news['id'] in test_ids:
    split = 'test'
  else:
    split = None

  rows_fakebr.append({
    'id': news['id'],
    'text': news['text'],
    'label': 'fake',
    'origin': 'human',
    'source': 'fake.br',
    'split': split,
    'llm_explanation': None
  })

In [15]:
rows_fakebr_llm = []

for item in train_fakebr_llm_raw:
  rows_fakebr_llm.append({
    'id': item['id'], 'text': item['original_news'],
    'label': 'true', 'origin': 'human', 'source': 'fake.br-llm',
    'split': 'train', 'llm_explanation': None
  })

  rows_fakebr_llm.append({
    'id': item['id'], 'text': item['synthetic_news'],
    'label': 'fake', 'origin': 'synthetic', 'source': 'fake.br-llm',
    'split': 'train', 'llm_explanation': item['llm_changes']
  })

for item in test_fakebr_llm_raw:
  rows_fakebr_llm.append({
    'id': item['id'], 'text': item['original_news'],
    'label': 'true', 'origin': 'human', 'source': 'fake.br-llm',
    'split': 'test', 'llm_explanation': None
  })

  rows_fakebr_llm.append({
    'id': item['id'], 'text': item['synthetic_news'],
    'label': 'fake', 'origin': 'synthetic', 'source': 'fake.br-llm',
    'split': 'test', 'llm_explanation': item['llm_changes']
  })

## 6. Validação do DataFrame

Conferência de contagens e amostras do DataFrame final para garantir que os dados foram carregados corretamente.

In [16]:
df_news = pd.DataFrame(rows_fakebr + rows_fakebr_llm)

print(f'Total de linhas: {len(df_news)}')
print(df_news['label'].value_counts())
print(df_news['origin'].value_counts())

Total de linhas: 10800
label
fake    7200
true    3600
Name: count, dtype: int64
origin
human        7200
synthetic    3600
Name: count, dtype: int64


In [17]:
df_news.head()

,id,text,label,origin,source,split,llm_explanation
0,2037,Garoto de 16 anos morre em escola do Paraná. E...,fake,human,fake.br,train,None
1,406,Bruno Gagliasso tira selfie com sua esposa bei...,fake,human,fake.br,train,None
2,3278,"Na Bahia, adolescente de 14 anos prevê a próp...",fake,human,fake.br,train,None
3,1787,Temer sem creme de avelã! Planalto decide canc...,fake,human,fake.br,train,None
4,310,"Uma nova geração (da Justiça) pede passagem: ""...",fake,human,fake.br,test,None


In [18]:
df_news.tail()

,id,text,label,origin,source,split,llm_explanation
10795,357,Estados Unidos e Coreia do Norte em vias de um...,fake,synthetic,fake.br-llm,test,1. **Adição de Fontes Anônimas**: A introdução...
10796,1539,O estranho caso do mergulhador cujo corpo come...,true,human,fake.br-llm,test,None
10797,1539,O incrível caso do mergulhador que se transfor...,fake,synthetic,fake.br-llm,test,1. **Exagero das deformações:** As descrições ...
10798,2219,Confronto de Renan com petistas e dispensa de ...,true,human,fake.br-llm,test,None
10799,2219,Exclusivo: Renan Calheiros expõe escândalo de ...,fake,synthetic,fake.br-llm,test,1. **Adição de uma revelação bombástica**: A i...


## 7. Salvamento do DataFrame

Exportação do DataFrame consolidado para uso nos notebooks seguintes.

In [19]:
os.makedirs('data/processed', exist_ok=True)
df_news.to_csv('data/processed/news_dataset.csv', index=False)